# Notebook 3 — SpaceX Web Scraping

**IBM Data Science Capstone** · Historical Wikipedia snapshot (9 June 2021).

This notebook extracts historical Falcon 9 / Falcon Heavy launch tables. It does not fabricate launch records. The source page and the scraping date are included for reproducibility.

In [1]:
%pip -q install requests beautifulsoup4 lxml pandas matplotlib

In [2]:
import io
import re
from pathlib import Path
from datetime import datetime, timezone

import requests
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display

STATIC_URL = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
headers = {"User-Agent": "Mozilla/5.0 (compatible; IBM-SpaceX-Capstone-Education/1.0)"}
response = requests.get(STATIC_URL, headers=headers, timeout=60)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
print("Page title:", soup.title.get_text(" ", strip=True) if soup.title else "missing")
print("HTTP:", response.status_code)
print("Retrieved UTC:", datetime.now(timezone.utc).isoformat())

Page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia
HTTP: 200
Retrieved UTC: 2026-10-29T22:32:54.101736+00:00


In [3]:
# Identify launch tables by their actual column headers (not fixed table indexes).
def normalize(s):
    return re.sub(r"\s+", " ", str(s)).strip()

launch_tables = []
for table in soup.select("table.wikitable"):
    header_rows = table.find_all("tr", limit=3)
    header_text = " ".join(normalize(th.get_text(" ", strip=True)).lower()
                           for row in header_rows for th in row.find_all("th"))
    if ("flight no" in header_text or "flight number" in header_text) and "payload" in header_text:
        launch_tables.append(table)

print("Matching launch tables:", len(launch_tables))
if not launch_tables:
    raise RuntimeError("No historical launch tables found; inspect the downloaded Wikipedia HTML.")

Matching launch tables: 9


In [4]:
# BeautifulSoup locates tables; pandas parses row/column spans into tabular records.
def flatten_columns(columns):
    flat = []
    for col in columns:
        parts = [normalize(c) for c in col] if isinstance(col, tuple) else [normalize(col)]
        parts = [p for p in parts if p and not p.startswith("Unnamed:")]
        flat.append(parts[-1] if parts else "Unknown")
    return flat

frames = []
for idx, table in enumerate(launch_tables, start=1):
    try:
        parsed = pd.read_html(io.StringIO(str(table)), displayed_only=False)[0]
        parsed.columns = flatten_columns(parsed.columns)
        parsed["SourceTable"] = idx
        frames.append(parsed)
    except (ValueError, ImportError) as ex:
        print(f"Skipping table {idx}: {ex}")

if not frames:
    raise RuntimeError("No HTML tables could be parsed.")

raw = pd.concat(frames, ignore_index=True, sort=False)
print("Extracted table shape:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head(10))

Extracted table shape: (259, 12)
Columns: ['Flight No.', 'Date and time (UTC)', 'Version, Booster [b]', 'Launch site', 'Payload[c]', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing', 'SourceTable', 'Version, Booster[b]']


,Flight No.,Date and time (UTC),"Version, Booster [b]",Launch site,Payload[c],Payload mass,Orbit,Customer,Launch outcome,Booster landing,SourceTable,"Version, Booster[b]"
0,1,"4 June 2010, 18:45",F9 v1.0[7] B0003.1[8],"CCAFS, SLC-40",Dragon Spacecraft Qualification Unit,NaN,LEO,SpaceX,Success,Failure[9][10] (parachute),1,NaN
1,1,First flight of Falcon 9 v1.0.[11] Used a boil...,First flight of Falcon 9 v1.0.[11] Used a boil...,First flight of Falcon 9 v1.0.[11] Used a boil...,First flight of Falcon 9 v1.0.[11] Used a boil...,First flight of Falcon 9 v1.0.[11] Used a boil...,First flight of Falcon 9 v1.0.[11] Used a boil...,First flight of Falcon 9 v1.0.[11] Used a boil...,First flight of Falcon 9 v1.0.[11] Used a boil...,First flight of Falcon 9 v1.0.[11] Used a boil...,1,NaN
2,2,"8 December 2010, 15:43[13]",F9 v1.0[7] B0004.1[8],"CCAFS, SLC-40",Dragon demo flight C1 (Dragon C101),NaN,LEO (ISS),".mw-parser-output .plainlist ol,.mw-parser-out...",Success[9],Failure[9][14] (parachute),1,NaN
3,2,"Maiden flight of Dragon capsule, consisting of...","Maiden flight of Dragon capsule, consisting of...","Maiden flight of Dragon capsule, consisting of...","Maiden flight of Dragon capsule, consisting of...","Maiden flight of Dragon capsule, consisting of...","Maiden flight of Dragon capsule, consisting of...","Maiden flight of Dragon capsule, consisting of...","Maiden flight of Dragon capsule, consisting of...","Maiden flight of Dragon capsule, consisting of...",1,NaN
4,3,"22 May 2012, 07:44[17]",F9 v1.0[7] B0005.1[8],"CCAFS, SLC-40",Dragon demo flight C2+[18] (Dragon C102),"525 kg (1,157 lb)[19]",LEO (ISS),NASA (COTS),Success[20],No attempt,1,NaN
5,3,Dragon spacecraft demonstrated a series of tes...,Dragon spacecraft demonstrated a series of tes...,Dragon spacecraft demonstrated a series of tes...,Dragon spacecraft demonstrated a series of tes...,Dragon spacecraft demonstrated a series of tes...,Dragon spacecraft demonstrated a series of tes...,Dragon spacecraft demonstrated a series of tes...,Dragon spacecraft demonstrated a series of tes...,Dragon spacecraft demonstrated a series of tes...,1,NaN
6,4,"8 October 2012, 00:35[21]",F9 v1.0[7] B0006.1[8],"CCAFS, SLC-40",SpaceX CRS-1[22] (Dragon C103),"4,700 kg (10,400 lb)",LEO (ISS),NASA (CRS),Success,No attempt,1,NaN
7,4,"8 October 2012, 00:35[21]",F9 v1.0[7] B0006.1[8],"CCAFS, SLC-40",Orbcomm-OG2[23],172 kg (379 lb)[24],LEO,Orbcomm,Partial failure[25],No attempt,1,NaN
8,4,"CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...","CRS-1 was successful, but the secondary payloa...",1,NaN
9,5,"1 March 2013, 15:10",F9 v1.0[7] B0007.1[8],"CCAFS, SLC-40",SpaceX CRS-2[22] (Dragon C104),"4,877 kg (10,752 lb)",LEO (ISS),NASA (CRS),Success,No attempt,1,NaN


In [5]:
# Check structural variation before making conclusions.
print("Non-null values per column:")
display(raw.notna().sum().sort_values(ascending=False).to_frame("NonNullCount"))

# Keep raw data, including source column names, to avoid silently changing historical records.
out = Path("spacex_web_scraped.csv")
raw.to_csv(out, index=False, encoding="utf-8")
print("Saved:", out.resolve())
print("Rows saved:", len(raw))

Non-null values per column:


,NonNullCount
SourceTable,259
Flight No.,257
Launch site,257
Date and time (UTC),257
Booster landing,257
Payload[c],257
Customer,257
Orbit,257
Launch outcome,257
Payload mass,255


Saved: /content/spacex_web_scraped.csv
Rows saved: 259


In [6]:
# Optional exploratory overview: assess year coverage from date-like columns.
possible_dates = [c for c in raw.columns if "date" in str(c).lower()]
print("Date-related columns:", possible_dates)
if possible_dates:
    print("Sample date values:")
    display(raw[possible_dates[0]].head(10).to_frame())

# Download file to upload into the data/ folder of your GitHub repository.
try:
    from google.colab import files
    files.download("spacex_web_scraped.csv")
except ImportError:
    print("Outside Google Colab: retrieve spacex_web_scraped.csv from the notebook directory.")

Date-related columns: ['Date and time (UTC)']
Sample date values:


,Date and time (UTC)
0,"4 June 2010, 18:45"
1,First flight of Falcon 9 v1.0.[11] Used a boil...
2,"8 December 2010, 15:43[13]"
3,"Maiden flight of Dragon capsule, consisting of..."
4,"22 May 2012, 07:44[17]"
5,Dragon spacecraft demonstrated a series of tes...
6,"8 October 2012, 00:35[21]"
7,"8 October 2012, 00:35[21]"
8,"CRS-1 was successful, but the secondary payloa..."
9,"1 March 2013, 15:10"


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpretation and limitations

- This is a **historical snapshot** of Wikipedia, not a live 2026 dataset.
- This notebook preserves extracted raw table columns. Date formats, merged header names, and supplementary rows may require cleaning in a separate step.
- Report the observed row count and any parsing exclusions; do not presume it equals the 90 observations in the *API-derived* IBM ML dataset.
- GitHub: commit both the executed `.ipynb` and its output CSV.